In [1]:
import pandas as pd 
from pathlib import Path 

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "DataCoSupplyChainDataset.csv"

df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print(df.shape)

(180519, 53)


In [2]:
df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Email',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Password',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'Product Card Id',
 'Product Category Id',
 'Product Description',
 'Product Image',
 'Product Name',
 'Product P

In [3]:
pd.set_option("display.max_columns", None)

df.head(2)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class


In [4]:
DATE_COLUMNS = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in DATE_COLUMNS:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [5]:
print(df["order date (DateOrders)"].dtype)
print(df["shipping date (DateOrders)"].dtype)

datetime64[us]
datetime64[us]


In [ ]:
TARGET = "Late_delivery_risk"

TECHNICAL_COLUMNS = [
    "Customer Id",
    "Order Id",
    "order date (DateOrders)",
]

MODEL_FEATURES = [
    "Type",
    "Category Id",
    "Customer Segment",
    "Customer State",
    "Department Name",
    "Order Country",
    "Order Item Discount",
    "Order Item Quantity",
    "Order Region",
    "Product Name",
    "Shipping Mode",
]

KEEP_COLUMNS = MODEL_FEATURES + TECHNICAL_COLUMNS + [TARGET]

In [ ]:
df_clean = df[KEEP_COLUMNS].copy()

print(df_clean.shape)
print(df_clean.columns.tolist())

(180519, 15)
['Type', 'Category Id', 'Customer Segment', 'Customer State', 'Department Name', 'Order Country', 'Order Item Discount', 'Order Item Quantity', 'Order Region', 'Product Name', 'Shipping Mode', 'Customer Id', 'Order Id', 'order date (DateOrders)', 'Late_delivery_risk']


In [9]:
INVALID_CUSTOMER_STATES = ["91732", "95758"]

matches = []

for col in df.columns:
    normalized = (
        df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    for value in INVALID_CUSTOMER_STATES:
        count = (normalized == value).sum()

        if count > 0:
            matches.append({
                "value": value,
                "column": col,
                "count": count
            })

pd.DataFrame(matches)

,value,column,count
0,91732,Customer State,1
1,95758,Customer State,2
2,91732,Customer Zipcode,126
3,95758,Customer Zipcode,113
4,91732,Order Item Id,1
5,95758,Order Item Id,1


In [10]:
zip_state_check = (
    df[
        df["Customer Zipcode"].isin([91732, 95758, 91732.0, 95758.0])
    ][
        ["Customer Zipcode", "Customer State"]
    ]
    .drop_duplicates()
    .sort_values(["Customer Zipcode", "Customer State"])
)

zip_state_check

,Customer Zipcode,Customer State
1738,91732.0,CA
1755,95758.0,CA


In [11]:
INVALID_STATE_MAP = {
    "91732": "CA",
    "95758": "CA",
}

df_clean["Customer State"] = (
    df_clean["Customer State"]
    .replace(INVALID_STATE_MAP)
)

In [12]:
print(
    df_clean["Customer State"]
    .isin(["91732", "95758"])
    .sum()
)

0


In [13]:
print("Shape:", df_clean.shape)
print("\nMissing values:")
print(df_clean.isna().sum())

print("\nData types:")
print(df_clean.dtypes)

Shape: (180519, 15)

Missing values:
Type                       0
Category Id                0
Customer Segment           0
Customer State             0
Department Name            0
Order Country              0
Order Item Discount        0
Order Item Quantity        0
Order Region               0
Product Name               0
Shipping Mode              0
Customer Id                0
Order Id                   0
order date (DateOrders)    0
Late_delivery_risk         0
dtype: int64

Data types:
Type                                  str
Category Id                         int64
Customer Segment                      str
Customer State                        str
Department Name                       str
Order Country                         str
Order Item Discount               float64
Order Item Quantity                 int64
Order Region                          str
Product Name                          str
Shipping Mode                         str
Customer Id                         int6

In [14]:
df_clean["Category Id"] = df_clean["Category Id"].astype("category")

In [15]:
print(df_clean["Category Id"].dtype)

category


In [16]:
df_clean["Category Id"].head(5)

0    73
1    73
2    73
3    73
4    73
Name: Category Id, dtype: category
Categories (51, int64): [2, 3, 4, 5, ..., 73, 74, 75, 76]

In [28]:
df[["Category Id", "Category Name"]].head(5)

,Category Id,Category Name
0,73,Sporting Goods
1,73,Sporting Goods
2,73,Sporting Goods
3,73,Sporting Goods
4,73,Sporting Goods


In [30]:
CATEGORICAL_FEATURES = [
    "Type",
    "Category Id",
    "Customer Segment",
    "Customer State",
    "Department Name",
    "Order Country",
    "Order Region",
    "Product Name",
    "Shipping Mode",
]

In [31]:
for col in CATEGORICAL_FEATURES:
    df_clean[col] = df_clean[col].astype("category")

In [32]:
print(df_clean[CATEGORICAL_FEATURES].dtypes)

Type                category
Category Id         category
Customer Segment    category
Customer State      category
Department Name     category
Order Country       category
Order Region        category
Product Name        category
Shipping Mode       category
dtype: object


In [33]:
print(df_clean.dtypes)

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nShape:", df_clean.shape)

Type                             category
Category Id                      category
Customer Segment                 category
Customer State                   category
Department Name                  category
Order Country                    category
Order Item Discount               float64
Order Item Quantity                 int64
Order Region                     category
Product Name                     category
Shipping Mode                    category
Customer Id                         int64
Order Id                            int64
order date (DateOrders)    datetime64[us]
Late_delivery_risk                  int64
dtype: object

Missing values:
Type                       0
Category Id                0
Customer Segment           0
Customer State             0
Department Name            0
Order Country              0
Order Item Discount        0
Order Item Quantity        0
Order Region               0
Product Name               0
Shipping Mode              0
Customer Id         

In [34]:
order_date_consistency = (
    df_clean.groupby("Order Id")["order date (DateOrders)"]
    .nunique()
)

print(
    "Orders with multiple order dates:",
    (order_date_consistency > 1).sum()
)

Orders with multiple order dates: 0


In [35]:
shipping_date_consistency = (
    df.groupby("Order Id")["shipping date (DateOrders)"]
    .nunique()
)

print(
    "Orders with multiple shipping dates:",
    (shipping_date_consistency > 1).sum()
)

Orders with multiple shipping dates: 0


In [36]:
within_order_variation = (
    df_clean.groupby("Order Id")[MODEL_FEATURES]
    .nunique()
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

within_order_variation

Order Item Discount    45780
Product Name           44578
Category Id            44563
Department Name        41471
Order Item Quantity    38816
Type                       0
Customer Segment           0
Customer State             0
Order Country              0
Order Region               0
Shipping Mode              0
dtype: int64

In [37]:
order_item_structure = (
    df_clean.groupby("Order Id")
    .agg(
        unique_products=("Product Name", "nunique"),
        unique_categories=("Category Id", "nunique"),
        unique_departments=("Department Name", "nunique"),
    )
)

print(order_item_structure.describe())

       unique_products  unique_categories  unique_departments
count     65752.000000       65752.000000        65752.000000
mean          2.429782           2.426132            2.002129
std           1.257508           1.254356            0.935964
min           1.000000           1.000000            1.000000
25%           1.000000           1.000000            1.000000
50%           2.000000           2.000000            2.000000
75%           3.000000           3.000000            3.000000
max           5.000000           5.000000            5.000000


In [38]:
for col in [
    "unique_products",
    "unique_categories",
    "unique_departments",
]:
    print(f"\n{col}")
    print(order_item_structure[col].value_counts().sort_index())


unique_products
unique_products
1    21174
2    14174
3    15102
4    11575
5     3727
Name: count, dtype: int64

unique_categories
unique_categories
1    21189
2    14204
3    15159
4    11551
5     3649
Name: count, dtype: int64

unique_departments
unique_departments
1    24281
2    21318
3    16066
4     3906
5      181
Name: count, dtype: int64


## aggregrate items of orders that have more than on items in an order

In [39]:
order_level_df = (
    df_clean
    .groupby("Order Id", as_index=False)
    .agg({
        "Type": "first",
        "Customer Segment": "first",
        "Customer State": "first",
        "Order Country": "first",
        "Order Region": "first",
        "Shipping Mode": "first",
        "Customer Id": "first",
        "order date (DateOrders)": "first",
        "Late_delivery_risk": "first",

        "Order Item Quantity": "sum",
        "Order Item Discount": "sum",
        "Product Name": "nunique",
        "Category Id": "nunique",
        "Department Name": "nunique",
    })
)

In [43]:
order_level_df = order_level_df.rename(
    columns={
        "Order Item Quantity": "total_quantity",
        "Order Item Discount": "total_discount",
        "Product Name": "num_unique_products",
        "Category Id": "num_unique_categories",
        "Department Name": "num_unique_departments",
    }
)

In [44]:
print(order_level_df.shape)
print(order_level_df.head())

(65752, 15)
   Order Id     Type Customer Segment Customer State Order Country  \
0         1     CASH         Consumer             NC        México   
1         2  PAYMENT         Consumer             IL      Colombia   
2         4     CASH      Home Office             TX      Colombia   
3         5    DEBIT         Consumer             PR      Colombia   
4         7    DEBIT         Consumer             FL        Brasil   

      Order Region   Shipping Mode  Customer Id order date (DateOrders)  \
0  Central America  Standard Class        11599     2015-01-01 00:00:00   
1    South America  Standard Class          256     2015-01-01 00:21:00   
2    South America  Standard Class         8827     2015-01-01 01:03:00   
3    South America  Standard Class        11318     2015-01-01 01:24:00   
4    South America    Second Class         4530     2015-01-01 02:06:00   

   Late_delivery_risk  total_quantity  total_discount  num_unique_products  \
0                   0               1 

In [45]:
print("Unique orders:", order_level_df["Order Id"].nunique())
print("Rows:", len(order_level_df))

print("\nMissing values:")
print(order_level_df.isna().sum())

print("\nTarget distribution:")
print(order_level_df["Late_delivery_risk"].value_counts(normalize=True))

Unique orders: 65752
Rows: 65752

Missing values:
Order Id                   0
Type                       0
Customer Segment           0
Customer State             0
Order Country              0
Order Region               0
Shipping Mode              0
Customer Id                0
order date (DateOrders)    0
Late_delivery_risk         0
total_quantity             0
total_discount             0
num_unique_products        0
num_unique_categories      0
num_unique_departments     0
dtype: int64

Target distribution:
Late_delivery_risk
1    0.548242
0    0.451758
Name: proportion, dtype: float64


In [46]:
print("Min total quantity:", order_level_df["total_quantity"].min())
print("Max total quantity:", order_level_df["total_quantity"].max())

print("Min unique products:", order_level_df["num_unique_products"].min())
print("Max unique products:", order_level_df["num_unique_products"].max())

Min total quantity: 1
Max total quantity: 24
Min unique products: 1
Max unique products: 5


### Cleaning Transformations to Productionize

- Parse `order date (DateOrders)` as datetime.
- Replace invalid `Customer State` ZIP-like values with `CA`.
- Keep only approved model/technical/target columns.
- Treat categorical features as categorical (Cast `Category Id` to categorical type).
- Learn rare-category mappings from training data only.
- Remove technical columns before model fitting.

# save the processed data on the dta folder as processed data

In [48]:
pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 11.7 MB/s  0:00:03m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [50]:
INTERIM_PATH = PROJECT_ROOT / "data" / "interim" / "order_level_clean.csv"

order_level_df.to_csv(
    INTERIM_PATH,
    index=False
)